# ferrite-tpu on a real TPU

Runs the whole project on the TPU attached to this notebook: the test-suite (real Mosaic
lowering, no interpreter), the kernel benchmarks with roofline numbers, two short training
runs (Pallas kernels vs XLA), the serving benchmark and a profiler trace. Results land in
`ferrite-tpu/results/<device>/` and are zipped at the end so they can be committed to the repo.

**Kaggle:** Settings -> Accelerator -> *TPU VM v3-8* (or v5e-8). Needs phone verification and
internet enabled. **Colab:** Runtime -> Change runtime type -> TPU. Expect 15-25 minutes end to end.


In [ ]:
REPO_URL = "https://github.com/anxnd2211/ferrite-tpu.git"
import os, subprocess
if not os.path.isdir("ferrite-tpu"):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, "ferrite-tpu"], check=True)
%pip install -q -U "jax[tpu]"
%pip install -q -e "./ferrite-tpu[dev]"


In [ ]:
import os, sys, json
# the editable install is only visible to new interpreters, so add the source tree for this kernel
sys.path.insert(0, os.path.abspath("ferrite-tpu/src"))
import jax, jax.numpy as jnp
print(jax.__version__, jax.devices())
assert jax.default_backend() == "tpu", "no TPU attached: fix the accelerator setting and restart the runtime"
from ferrite import platform
print(json.dumps(platform.describe(), indent=2))
sys.path.insert(0, os.path.abspath("ferrite-tpu/benchmarks"))
from _common import device_slug
DEV = device_slug(); RES = f"results/{DEV}"
print("results ->", RES)


## 1. Tests on real hardware
Same tests as CI, but the kernels now go through the Mosaic compiler and run on the TPU.

In [ ]:
!cd ferrite-tpu && python -m pytest -q -x -p no:cacheprovider

## 2. Kernel benchmarks + roofline
Each kernel against its XLA baseline (and JAX's own Pallas flash attention), with achieved TFLOP/s, GB/s and percent of the roofline for this chip.

In [ ]:
!cd ferrite-tpu && python benchmarks/bench_kernels.py --iters 20

## 3. Training: Pallas kernels vs XLA
13M-parameter decoder on byte-level TinyShakespeare, data-parallel over all cores. Same seed, same data order; compare step time, tokens/s and MFU.

In [ ]:
!cd ferrite-tpu && python -m ferrite.model.train --steps 300 --batch 64 --seq 512 --attention pallas --norm pallas --out {RES}/train_pallas.json --save-params checkpoints/pallas.npz
!cd ferrite-tpu && python -m ferrite.model.train --steps 300 --batch 64 --seq 512 --attention xla --norm xla --out {RES}/train_xla.json
!cd ferrite-tpu && python -m ferrite.model.train --steps 300 --batch 64 --seq 512 --attention pallas --norm pallas --linear pallas --out {RES}/train_pallas_all.json


## 4. Serving: AOT-compiled greedy decoding
TTFT, inter-token latency and tokens/s at several batch sizes, with Pallas vs XLA prefill, using the checkpoint from step 3.

In [ ]:
!cd ferrite-tpu && python benchmarks/bench_serve.py --params checkpoints/pallas.npz --seq 512 --batches 1,8,32 --prompt-len 100 --new-tokens 128

## 5. Profiler trace
Captures a device trace of the flash-attention kernel (forward + backward) next to the XLA baseline. Open the folder in TensorBoard (`tensorboard-plugin-profile`) or upload the `.xplane.pb` to https://ui.perfetto.dev to see each `tpu_custom_call` with its measured duration.

In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath("ferrite-tpu/src"))
os.chdir("ferrite-tpu")
from ferrite.kernels.flash_attention import BlockSizes, flash_attention
from ferrite.profiling import trace
import jax, jax.numpy as jnp
b, h, s, d = 4, 8, 2048, 128
ks = jax.random.split(jax.random.key(0), 3)
q, k, v = (jax.random.normal(kk, (b, h, s, d), jnp.float32).astype(jnp.bfloat16) for kk in ks)
def xla_attn(q, k, v):
    t = lambda a: jnp.swapaxes(a, 1, 2)
    return t(jax.nn.dot_product_attention(t(q), t(k), t(v), is_causal=True, implementation="xla"))
f_pallas = jax.jit(jax.grad(lambda q, k, v: jnp.sum(flash_attention(q, k, v, causal=True, block_sizes=BlockSizes.for_device()).astype(jnp.float32)), argnums=(0, 1, 2)))
f_xla = jax.jit(jax.grad(lambda q, k, v: jnp.sum(xla_attn(q, k, v).astype(jnp.float32)), argnums=(0, 1, 2)))
for f in (f_pallas, f_xla):
    jax.block_until_ready(f(q, k, v))  # compile outside the trace
with trace("traces/flash_attention_vs_xla"):
    for _ in range(5):
        jax.block_until_ready(f_pallas(q, k, v))
        jax.block_until_ready(f_xla(q, k, v))
print(sorted(os.listdir("traces/flash_attention_vs_xla")))
os.chdir("..")


## 6. Report

In [ ]:
!cd ferrite-tpu && python benchmarks/make_report.py results > /dev/null
from IPython.display import Markdown, display
display(Markdown(open(f"ferrite-tpu/{RES}/REPORT.md").read()))


In [ ]:
!cd ferrite-tpu && zip -qr ../ferrite_results.zip results traces && ls -la ../ferrite_results.zip
print("Download ferrite_results.zip (Output panel), unzip into the repo, commit results/<device>/ and traces/.")
